# Cross-nested logit (CNL)

Estimate a CNL model in which each alternative has fixed allocation weights across public and private nests.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import (
    Beta,
    ChoiceDataset,
    CrossNest,
    CrossNestedLogit,
    UtilitySpec,
)
from torchdcm.datasets import make_swissmetro_like

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec

rng = np.random.default_rng(130)
frame = make_swissmetro_like(n_obs=600, seed=13)
frame[["avail_train", "avail_sm", "avail_car"]] = 1
initial_data = ChoiceDataset.from_wide(
    frame,
    alternatives=["TRAIN", "SM", "CAR"],
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)

dgp_spec = UtilitySpec()
dgp_spec.utility(
    "TRAIN",
    Beta("ASC_TRAIN_DGP", init=0.30)
    + Beta("B_TIME_DGP", init=-0.03) * "time"
    + Beta("B_COST_DGP", init=-0.05) * "cost",
)
dgp_spec.utility(
    "SM",
    Beta("B_TIME_DGP", init=-0.03) * "time"
    + Beta("B_COST_DGP", init=-0.05) * "cost",
)
dgp_spec.utility(
    "CAR",
    Beta("ASC_CAR_DGP", init=0.20)
    + Beta("B_TIME_DGP", init=-0.03) * "time"
    + Beta("B_COST_DGP", init=-0.05) * "cost",
)
dgp = CrossNestedLogit(
    dgp_spec,
    {
        "PUBLIC": CrossNest(
            {"TRAIN": 0.80, "SM": 0.80, "CAR": 0.20},
            init=0.65,
        ),
        "PRIVATE": CrossNest(
            {"TRAIN": 0.20, "SM": 0.20, "CAR": 0.80},
            init=0.90,
        ),
    },
)
true_params = torch.tensor([0.30, -0.03, -0.05, 0.20, 0.65, 0.90])
probabilities = (
    dgp.predict_proba(initial_data, true_params)
    .reshape(-1, 3)
    .detach()
    .cpu()
    .numpy()
)
alternatives = np.array(["TRAIN", "SM", "CAR"])
frame["choice"] = [
    rng.choice(alternatives, p=row) for row in probabilities
]
data = ChoiceDataset.from_wide(
    frame,
    alternatives=alternatives.tolist(),
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)
spec = make_utility_spec()


In [3]:
nests = {
    "PUBLIC": CrossNest(
        {"TRAIN": 0.80, "SM": 0.80, "CAR": 0.20},
        init=0.80,
    ),
    "PRIVATE": CrossNest(
        {"TRAIN": 0.20, "SM": 0.20, "CAR": 0.80},
        init=0.95,
    ),
}
model = CrossNestedLogit(spec, nests, device=device, max_iter=80)
result = model.fit(data)
display(HTML(result.report().to_html()))


Model family,CrossNestedLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,80
Line search,strong_wolfe
